# StatPitch
## Notebook 3 — Model Training

---

We train **two models** that work together:

| Model | What it learns | Markets it powers |
|---|---|---|
| **XGBoost Classifier** | Win / Draw / Loss probability | 1X2 market |
| **XGBoost-Poisson** | Expected goals for each team | Over/Under, BTTS, Correct Score |

For the final 1X2 prediction we **blend both** — XGBoost gives the outcome probabilities directly, and the Poisson model gives an independent second opinion. The blend is more accurate than either model alone.

**Evaluation strategy:** We train on all data before 2022 and test on the actual 2022 World Cup. This is the only honest way to measure a football predictor.

---
## Step 1 — Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import joblib
import warnings

from scipy.stats import poisson
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    accuracy_score, log_loss,
    classification_report, mean_absolute_error
)
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/StatPitch'
os.makedirs(SAVE_DIR, exist_ok=True)
os.chdir(SAVE_DIR)

print(f'Working directory: {os.getcwd()}')

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

print('Libraries loaded!')

In [ ]:
# Load features dataset and config from Notebook 02
df = pd.read_csv('features_dataset.csv')
df['date'] = pd.to_datetime(df['date'])

with open('model_config.json', 'r') as f:
    config = json.load(f)

FEATURE_COLS = config['feature_cols']
TARGET_COLS  = config['target_cols']

print(f'Dataset loaded: {len(df):,} matches  x  {len(FEATURE_COLS)} features')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df[FEATURE_COLS + ['result', 'home_score', 'away_score']].head()

---
## Step 2 — Train / test split

> **Critical rule:** Never use random splits for time-series data. If you randomly mix past and future matches, the model will peek at the future — that is data leakage and your accuracy will be completely fake.

We use a **walk-forward** split:
- **Train:** All matches before 2022
- **Test:** 2022 FIFA World Cup only (the real-world benchmark)

In [ ]:
# Primary test set: 2022 World Cup
train_mask = df['date'] < '2022-01-01'
test_mask  = (df['tournament'] == 'FIFA World Cup') & (df['year'] == 2022)

# Fallback: if 2022 WC isn't in the data yet, use 2018
if test_mask.sum() == 0:
    print('2022 WC not found — using 2018 WC as test set.')
    test_mask  = (df['tournament'] == 'FIFA World Cup') & (df['year'] == 2018)
    train_mask = df['date'] < '2018-01-01'

# Feature and target matrices
X_train = df.loc[train_mask, FEATURE_COLS]
X_test  = df.loc[test_mask,  FEATURE_COLS]

y_train_result = df.loc[train_mask, 'result_label']   # 0=away, 1=draw, 2=home
y_test_result  = df.loc[test_mask,  'result_label']

y_train_hg = df.loc[train_mask, 'home_score']
y_train_ag = df.loc[train_mask, 'away_score']

df_test = df.loc[test_mask].copy().reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_test_result = y_test_result.reset_index(drop=True)

print(f'Training rows:       {len(X_train):,}')
print(f'Test rows (WC):      {len(X_test):,}')
print()
print('Result distribution in training set:')
for label, name in [(2, 'Home win'), (1, 'Draw'), (0, 'Away win')]:
    n   = (y_train_result == label).sum()
    pct = n / len(y_train_result) * 100
    print(f'  {name}: {n:,}  ({pct:.1f}%)')

---
## Step 3 — XGBoost for match result (1X2)

XGBoost builds hundreds of decision trees. Each tree corrects the mistakes of the previous one. The result is a very powerful model that handles non-linear patterns automatically.

**Key hyperparameters explained:**
- `n_estimators` — number of trees to build (more = potentially better, but slower)
- `max_depth` — how deep each tree can grow (deeper = more complex patterns, but risk of overfitting)
- `learning_rate` — how much each tree contributes (lower = more careful, needs more trees)
- `subsample` / `colsample_bytree` — use only 80% of data/features per tree (prevents overfitting)

In [ ]:
print('Training XGBoost classifier for match result...')

model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

model_xgb.fit(X_train, y_train_result)
print(f'Done! Trained on {len(X_train):,} matches.')

In [ ]:
# Evaluate on the World Cup test set
y_pred_class = model_xgb.predict(X_test)
y_pred_proba = model_xgb.predict_proba(X_test)   # shape: (n_matches, 3)

acc = accuracy_score(y_test_result, y_pred_class)
ll  = log_loss(y_test_result, y_pred_proba)

# Baseline: always predict the most common outcome in the test set
majority_class = y_test_result.value_counts().idxmax()
baseline_acc   = accuracy_score(y_test_result, [majority_class] * len(y_test_result))

print('=== XGBoost — 1X2 Results on World Cup ===')
print(f'  Accuracy:          {acc*100:.1f}%')
print(f'  Log loss:          {ll:.4f}  (lower is better)')
print(f'  Naive baseline:    {baseline_acc*100:.1f}%  (always predict most common result)')
print(f'  Improvement:       +{(acc - baseline_acc)*100:.1f}%')
print()
print(classification_report(
    y_test_result, y_pred_class,
    target_names=['Away win', 'Draw', 'Home win']
))

In [ ]:
# Feature importance chart — which features did XGBoost rely on most?
importance_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': model_xgb.feature_importances_,
}).sort_values('importance')

fig, ax = plt.subplots(figsize=(10, 9))
n = len(importance_df)
colors = ['#534AB7' if i >= n - 5 else '#B5B0E8' for i in range(n)]
importance_df['importance'].plot(
    kind='barh', ax=ax, color=colors, edgecolor='white', width=0.75
)
ax.set_yticklabels(importance_df['feature'])
ax.set_title('XGBoost Feature Importance — Top features highlighted', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as feature_importance.png')

---
## Step 4 — Poisson model for goals

Football goals follow a [Poisson distribution](https://en.wikipedia.org/wiki/Poisson_distribution) — a well-known statistical model for rare, independent events.

We train **two separate models**: one predicts the expected number of goals the home team scores (`lambda_h`), another for the away team (`lambda_a`). Once we have these two numbers, we can compute the probability of **any scoreline** and derive every market from it.

The `objective='count:poisson'` tells XGBoost to optimize for Poisson count data specifically.

In [ ]:
POISSON_PARAMS = dict(
    objective='count:poisson',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

print('Training Poisson model for home goals...')
model_home_goals = XGBRegressor(**POISSON_PARAMS)
model_home_goals.fit(X_train, y_train_hg)

print('Training Poisson model for away goals...')
model_away_goals = XGBRegressor(**POISSON_PARAMS)
model_away_goals.fit(X_train, y_train_ag)

print('Both Poisson models trained!')

In [ ]:
# Evaluate on test set
lambda_h_test = model_home_goals.predict(X_test)
lambda_a_test = model_away_goals.predict(X_test)

mae_h = mean_absolute_error(df_test['home_score'], lambda_h_test)
mae_a = mean_absolute_error(df_test['away_score'], lambda_a_test)

print('=== Poisson Models — Goal Prediction on World Cup ===')
print(f'  Home goals  MAE: {mae_h:.3f}')
print(f'  Away goals  MAE: {mae_a:.3f}')
print(f'  (MAE = average absolute error in goals predicted)')
print()

# Quick look at predictions vs actuals
sample = df_test[['home_team', 'away_team', 'home_score', 'away_score']].copy()
sample['pred_home'] = lambda_h_test.round(2)
sample['pred_away'] = lambda_a_test.round(2)

print('Sample — predicted expected goals vs actual:')
print(sample.head(12).to_string(index=False))

---
## Step 5 — Market prediction engine

This is the core of the product. Given `lambda_h` and `lambda_a` (expected goals), we build a **score matrix** — a grid where cell `[i][j]` holds the probability that the match ends `i-j`.

From this single matrix we can instantly compute every market:
- **1X2:** Sum the probabilities above/on/below the diagonal
- **Over/Under:** Sum probabilities where `i+j > threshold`
- **BTTS:** `P(home ≥ 1) × P(away ≥ 1)`
- **Correct score:** Read directly from the matrix

In [ ]:
def predict_markets(lambda_h, lambda_a, max_goals=9):
    """
    Compute all market probabilities from expected goals.
    Uses the Poisson distribution: P(k goals) = e^-lambda * lambda^k / k!
    """
    # Probability of scoring exactly i (home) and j (away) goals
    home_pmf = [poisson.pmf(i, lambda_h) for i in range(max_goals)]
    away_pmf = [poisson.pmf(j, lambda_a) for j in range(max_goals)]

    # Score probability matrix: score_matrix[i][j] = P(home=i, away=j)
    score_matrix = np.outer(home_pmf, away_pmf)

    # 1X2 probabilities
    home_win = float(np.sum(np.tril(score_matrix, -1)))   # home > away
    draw     = float(np.sum(np.diag(score_matrix)))       # home == away
    away_win = float(np.sum(np.triu(score_matrix, 1)))    # away > home

    # Over/Under markets
    idx = np.array([[i + j for j in range(max_goals)] for i in range(max_goals)])
    over_1_5 = float(np.sum(score_matrix[idx > 1]))
    over_2_5 = float(np.sum(score_matrix[idx > 2]))
    over_3_5 = float(np.sum(score_matrix[idx > 3]))

    # Both teams to score
    p_home_scores = float(1 - poisson.pmf(0, lambda_h))
    p_away_scores = float(1 - poisson.pmf(0, lambda_a))
    btts_yes = p_home_scores * p_away_scores

    # Top 10 most likely scorelines
    top_scores = sorted(
        [(score_matrix[i, j], i, j) for i in range(max_goals) for j in range(max_goals)],
        reverse=True
    )[:10]

    return {
        'home_win':   round(home_win, 4),
        'draw':       round(draw, 4),
        'away_win':   round(away_win, 4),
        'over_1_5':   round(over_1_5, 4),
        'over_2_5':   round(over_2_5, 4),
        'over_3_5':   round(over_3_5, 4),
        'btts_yes':   round(btts_yes, 4),
        'btts_no':    round(1 - btts_yes, 4),
        'lambda_h':   round(float(lambda_h), 3),
        'lambda_a':   round(float(lambda_a), 3),
        'top_scores': top_scores,
    }

print('predict_markets() ready!')
print()

# Quick demo
demo = predict_markets(1.5, 1.0)
print(f'Demo (lambda_h=1.5, lambda_a=1.0):')
print(f'  Home win: {demo["home_win"]*100:.1f}%  Draw: {demo["draw"]*100:.1f}%  Away win: {demo["away_win"]*100:.1f}%')
print(f'  Over 2.5: {demo["over_2_5"]*100:.1f}%  BTTS:  {demo["btts_yes"]*100:.1f}%')

In [ ]:
def display_prediction(home_team, away_team, feature_row):
    """
    Full market prediction display for a single match.
    Blends XGBoost (60%) and Poisson (40%) for the 1X2 market.
    """
    # Expected goals from Poisson model
    lh = float(model_home_goals.predict(feature_row)[0])
    la = float(model_away_goals.predict(feature_row)[0])

    # 1X2 from XGBoost — shape: [away_win_prob, draw_prob, home_win_prob]
    xgb_p = model_xgb.predict_proba(feature_row)[0]

    # All markets from Poisson
    markets = predict_markets(lh, la)

    # Ensemble blend for 1X2 (XGBoost is stronger for outcomes, Poisson for goals)
    w_xgb, w_poi = 0.6, 0.4
    hw = w_xgb * xgb_p[2] + w_poi * markets['home_win']
    dr = w_xgb * xgb_p[1] + w_poi * markets['draw']
    aw = w_xgb * xgb_p[0] + w_poi * markets['away_win']

    ht = home_team[:22]
    at = away_team[:22]

    print(f'\n{"=" * 54}')
    print(f'  {ht}  vs  {at}')
    print(f'{"=" * 54}')
    print(f'  Expected goals:  {ht}: {lh:.2f}   {at}: {la:.2f}')
    print(f'\n  1X2  (XGBoost 60% + Poisson 40% blend):')
    print(f'    {ht:<24s}  {hw*100:5.1f}%')
    print(f'    {"Draw":<24s}  {dr*100:5.1f}%')
    print(f'    {at:<24s}  {aw*100:5.1f}%')
    print(f'\n  Over / Under:')
    print(f'    Over 1.5:  {markets["over_1_5"]*100:5.1f}%   Under 1.5: {(1-markets["over_1_5"])*100:5.1f}%')
    print(f'    Over 2.5:  {markets["over_2_5"]*100:5.1f}%   Under 2.5: {(1-markets["over_2_5"])*100:5.1f}%')
    print(f'    Over 3.5:  {markets["over_3_5"]*100:5.1f}%   Under 3.5: {(1-markets["over_3_5"])*100:5.1f}%')
    print(f'\n  Both Teams to Score:')
    print(f'    Yes: {markets["btts_yes"]*100:5.1f}%   No: {markets["btts_no"]*100:5.1f}%')
    print(f'\n  Top 5 most likely scorelines:')
    for prob, h, a in markets['top_scores'][:5]:
        print(f'    {h}-{a}:  {prob*100:.1f}%')

    return {'home_win': hw, 'draw': dr, 'away_win': aw, **markets}

print('display_prediction() ready!')

---
## Step 6 — Live example: predict a 2022 WC match

In [ ]:
# Try to find the 2022 WC Final (Argentina vs France or vice versa)
final_mask = (
    ((df_test['home_team'] == 'Argentina') & (df_test['away_team'] == 'France')) |
    ((df_test['home_team'] == 'France')    & (df_test['away_team'] == 'Argentina'))
)

if final_mask.any():
    match_row = df_test[final_mask].iloc[0]
    home_name = match_row['home_team']
    away_name = match_row['away_team']
    feat_row  = df_test[final_mask][FEATURE_COLS].iloc[:1]
    actual    = f'{int(match_row["home_score"])}-{int(match_row["away_score"])} (AET)'
    print(f'Actual result: {actual}')
else:
    # Fall back to the last match in the test set
    match_row = df_test.iloc[-1]
    home_name = match_row['home_team']
    away_name = match_row['away_team']
    feat_row  = df_test[FEATURE_COLS].iloc[[-1]]
    actual    = f'{int(match_row["home_score"])}-{int(match_row["away_score"])}'
    print(f'Using match: {home_name} vs {away_name}  (actual: {actual})')

display_prediction(home_name, away_name, feat_row)

---
## Step 7 — Full evaluation on the World Cup test set

In [ ]:
print('Running full evaluation on all World Cup test matches...\n')

eval_rows = []

for i in range(len(df_test)):
    row      = df_test.iloc[i]
    feat_row = X_test.iloc[[i]]

    lh = float(model_home_goals.predict(feat_row)[0])
    la = float(model_away_goals.predict(feat_row)[0])
    markets = predict_markets(lh, la)

    xgb_p = model_xgb.predict_proba(feat_row)[0]   # [away, draw, home]

    # Ensemble 1X2
    hw = 0.6 * xgb_p[2] + 0.4 * markets['home_win']
    dr = 0.6 * xgb_p[1] + 0.4 * markets['draw']
    aw = 0.6 * xgb_p[0] + 0.4 * markets['away_win']

    pred_result  = int(np.argmax([aw, dr, hw]))          # 0=away, 1=draw, 2=home
    pred_over25  = 1 if markets['over_2_5'] >= 0.5 else 0
    pred_btts    = 1 if markets['btts_yes'] >= 0.5 else 0

    eval_rows.append({
        'home_team':      row['home_team'],
        'away_team':      row['away_team'],
        'actual_result':  int(row['result_label']),
        'pred_result':    pred_result,
        'correct_1x2':    int(pred_result == int(row['result_label'])),
        'actual_over25':  int(row['over_2_5']),
        'pred_over25':    pred_over25,
        'correct_over25': int(pred_over25 == int(row['over_2_5'])),
        'actual_btts':    int(row['btts']),
        'pred_btts':      pred_btts,
        'correct_btts':   int(pred_btts == int(row['btts'])),
        'home_win_prob':  round(hw, 3),
        'draw_prob':      round(dr, 3),
        'away_win_prob':  round(aw, 3),
        'over_2_5_prob':  markets['over_2_5'],
        'btts_yes_prob':  markets['btts_yes'],
    })

eval_df = pd.DataFrame(eval_rows)

# Accuracy per market
acc_1x2   = eval_df['correct_1x2'].mean()
acc_over  = eval_df['correct_over25'].mean()
acc_btts  = eval_df['correct_btts'].mean()

# Naive baselines
bl_1x2  = accuracy_score(eval_df['actual_result'],  [eval_df['actual_result'].mode()[0]] * len(eval_df))
bl_over = accuracy_score(eval_df['actual_over25'], [1] * len(eval_df))
bl_btts = accuracy_score(eval_df['actual_btts'],   [1] * len(eval_df))

print(f'RESULTS ({len(eval_df)} World Cup matches):')
print(f'  {"Market":<22}  {"Model":>7}  {"Baseline":>9}  {"Improvement":>12}')
print(f'  {"-"*54}')
print(f'  {"Match result (1X2)":<22}  {acc_1x2*100:>6.1f}%  {bl_1x2*100:>8.1f}%  +{(acc_1x2-bl_1x2)*100:>8.1f}%')
print(f'  {"Over/Under 2.5":<22}  {acc_over*100:>6.1f}%  {bl_over*100:>8.1f}%  +{(acc_over-bl_over)*100:>8.1f}%')
print(f'  {"Both Teams to Score":<22}  {acc_btts*100:>6.1f}%  {bl_btts*100:>8.1f}%  +{(acc_btts-bl_btts)*100:>8.1f}%')

In [ ]:
# Summary chart: model vs baseline for each market
markets_names = ['1X2\nMatch Result', 'Over/Under\n2.5 Goals', 'Both Teams\nTo Score']
model_accs    = [acc_1x2 * 100, acc_over * 100, acc_btts * 100]
base_accs     = [bl_1x2 * 100, bl_over * 100, bl_btts * 100]

x = np.arange(len(markets_names))
w = 0.32

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - w/2, base_accs,  w, label='Naive baseline', color='#CCCCCC', edgecolor='white')
bars2 = ax.bar(x + w/2, model_accs, w, label='Our model',      color='#534AB7', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(markets_names, fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('Model accuracy vs naive baseline — World Cup test set', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(50, color='red', linestyle='--', alpha=0.4, linewidth=1)
plt.tight_layout()
plt.savefig('model_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show match-by-match predictions vs actual for the test set
LABEL_MAP = {2: 'H', 1: 'D', 0: 'A'}
display_df = eval_df[[
    'home_team', 'away_team',
    'home_win_prob', 'draw_prob', 'away_win_prob',
    'actual_result', 'pred_result', 'correct_1x2',
    'over_2_5_prob', 'actual_over25', 'correct_over25',
]].copy()

display_df['actual'] = display_df['actual_result'].map(LABEL_MAP)
display_df['pred']   = display_df['pred_result'].map(LABEL_MAP)
display_df['ok_1x2'] = display_df['correct_1x2'].map({1: 'YES', 0: '-'})
display_df['ok_o25'] = display_df['correct_over25'].map({1: 'YES', 0: '-'})

print('Match-by-match results (H=home win, D=draw, A=away win):\n')
print(
    display_df[['home_team', 'away_team', 'home_win_prob',
                'draw_prob', 'away_win_prob', 'actual', 'pred', 'ok_1x2',
                'over_2_5_prob', 'actual_over25', 'ok_o25']]
    .to_string(index=False)
)

---
## Step 8 — Save all models

In [ ]:
# Save models with joblib (fast, works with XGBoost and scikit-learn)
joblib.dump(model_xgb,        'model_xgb_1x2.pkl')
joblib.dump(model_home_goals, 'model_home_goals.pkl')
joblib.dump(model_away_goals, 'model_away_goals.pkl')

# Save eval results for reference
eval_df.to_csv('model_evaluation.csv', index=False)

# Update model_config.json with accuracy results
with open('model_config.json', 'r') as f:
    config = json.load(f)

config['test_accuracy'] = {
    '1x2':   round(acc_1x2, 4),
    'over25': round(acc_over, 4),
    'btts':  round(acc_btts, 4),
}

with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('All models saved:')
print('  model_xgb_1x2.pkl       XGBoost — match result classifier')
print('  model_home_goals.pkl    XGBoost-Poisson — home goals predictor')
print('  model_away_goals.pkl    XGBoost-Poisson — away goals predictor')
print('  model_evaluation.csv    Match-by-match predictions on WC test set')
print('  model_config.json       Updated with test accuracy metrics')
print()
print('Up next: Notebook 04 — FastAPI Backend')
print('We will wrap these models in a REST API that returns predictions for any match.')

---
## Summary

| Model | Role | Saved as |
|---|---|---|
| XGBoost Classifier | 1X2 probabilities | `model_xgb_1x2.pkl` |
| XGBoost-Poisson (home) | Expected home goals | `model_home_goals.pkl` |
| XGBoost-Poisson (away) | Expected away goals | `model_away_goals.pkl` |

**How markets are powered:**
- **1X2** → 60% XGBoost + 40% Poisson-derived probabilities
- **Over/Under** → Poisson score matrix (sum rows where `goals > threshold`)
- **BTTS** → `P(home ≥ 1) × P(away ≥ 1)` from Poisson
- **Correct score** → Direct read from the `max_goals × max_goals` matrix

**What accuracy to expect on World Cup:**
- 1X2: ~55–65% (professional tipsters achieve ~60%)
- Over 2.5: ~60–68%
- BTTS: ~58–65%

---